# Starter Notebook: Qwen 2B LoRA for Text-to-SVG (Kaggle)

This starter is built from the resources in `contest_docs`:
- Data resources: `contest_docs/03_Data_Design.md`
- Baseline and starter guidance: `contest_docs/05_Baselines_and_Starter_Notebooks.md`
- Kaggle implementation notes: `contest_docs/06_Kaggle_Implementation_Guide.md`

Goal: provide a practical scaffold for Qwen-2B-class fine-tuning + submission generation.

## Referenced Data and Docs

### Dataset resources from `contest_docs/03_Data_Design.md`
- `OmniSVG/MMSVG-Icon`
- `xingxm/SVGX-Core-250k`
- `xingxm/SVGX-SFT-1M` (recommended subset: `SVGX_SFT_GEN_51k.json`)
- `nyuuzyou/svgfind`
- `starvector/svg-icons`
- `thesantatitan/deepseek-svg-dataset`
- `InternSVG/SArena` (evaluation benchmark)

### Qwen 2B fine-tuning references from `contest_docs/05` and `contest_docs/06`
- Unsloth Qwen fine-tune docs: https://unsloth.ai/docs/models/qwen3.5/fine-tune
- Qwen3.5-2B Vision notebook: https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(2B)_Vision.ipynb

Note: this notebook is written as a reusable starter. You may need to adjust exact model IDs and column names to match the latest upstream datasets.

In [1]:
# Uncomment in a fresh Kaggle notebook environment.
#%pip install -q unsloth datasets trl transformers accelerate peft bitsandbytes pandas lxml

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

CUDA available: True
NVIDIA GeForce RTX 5070 Ti


In [3]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import ( LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training, PeftModel, )
from trl import SFTTrainer
#from datasets import concatenate_datasets, load_dataset

W0319 01:28:39.321000 18988 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Torch: {torch.__version__}")
print(torch.cuda.get_device_name(0))
print(f"CUDA available: {torch.cuda.is_available()}")

Torch: 2.10.0+cu130
NVIDIA GeForce RTX 5070 Ti
CUDA available: True


In [5]:
#these are models that can be tested in the future
#Qwen2.5-Coder-1.5B-Instruct-bnb-4bit
#"Qwen/Qwen2.5-Coder-1.5B-Instruct"

In [6]:
# Core training config.
# Keep runtime targets in line with contest_docs guidance (roughly <= 6-8 hours training).

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

CONFIG = {
    "max_seq_length": 2048,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "learning_rate": 2e-4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "save_steps": 200,
    "eval_steps": 100,
    "eval_ratio": 0.05,
    "output_dir": "./svg_lora_adapter",
    "debug_train_rows": 5000,   # set to None or comment out for full training
    "submission_path": "./submission.csv",
}

CONFIG

{'max_seq_length': 2048,
 'num_train_epochs': 1,
 'per_device_train_batch_size': 2,
 'gradient_accumulation_steps': 8,
 'learning_rate': 0.0002,
 'warmup_ratio': 0.05,
 'weight_decay': 0.01,
 'logging_steps': 20,
 'save_steps': 200,
 'eval_steps': 100,
 'eval_ratio': 0.05,
 'output_dir': './svg_lora_adapter',
 'debug_train_rows': 5000,
 'submission_path': './submission.csv'}

In [7]:
TRAIN_PATH = "kaggle/input/competitions/dl-spring-2026-svg-generation/train.csv"
TEST_PATH = "kaggle/input/competitions/dl-spring-2026-svg-generation/test.csv"
SAMPLE_SUB_PATH = "kaggle/input/competitions/dl-spring-2026-svg-generation/sample_submission.csv"

In [8]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)

print(train_df.shape)
print(test_df.shape)
print(sample_sub_df.shape)

print(train_df.head())
print(test_df.head())

(50000, 3)
(1000, 2)
(1000, 2)
                                           id  \
0            fd61e324e0cec5c11f337d0bfe2858c8   
1            999b3d4d5a860725bf9528910b5612f3   
2            1aaa84517819c25f783ae1c0cb337fc5   
3            919a7da8bd44dc7781dbe87383a268cc   
4  thesantatitan_deepseek-svg-dataset_0000581   

                                              prompt  \
0  The image features two orange squares with a m...   
1  A simple smiley face with a wide open mouth an...   
2  The image features a black-outlined icon of a ...   
3  The image displays a black icon with a photo-l...   
4  Generate svg code for an image that looks like...   

                                                 svg  
0  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
1  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
2  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
3  <svg xmlns="http://www.w3.org/2000/svg" viewBo...  
4  <svg width="24" height="24" viewBox="0 0 24 24...  
       

In [9]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask",
    "linearGradient", "radialGradient", "stop",
    "text", "tspan", "title", "desc", "style",
    "pattern", "marker", "filter"
}

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup with Canvas size 256 × 256 pixels and at most 16,000 characters. "
    "Return only SVG code with a single root <svg> element. "
    "Max path count must be less than 256. "
    "Use only allowed SVG tags and keep the output concise. "
)

In [10]:

def extract_svg(text: str) -> str:
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="48" fill="black"/>'
        '</svg>'
    )

def is_valid_svg(svg_text: str) -> bool:
    if not svg_text:
        return False

    # Be conservative. One of your docs mentioned 8000, another 16000.
    # Using 8000 is safer for debugging / submission safety.
    if len(svg_text) > 8000:
        return False

    if svg_text.count("<path") > 256:
        return False

    try:
        root = ET.fromstring(svg_text)
    except ET.ParseError:
        return False

    root_tag = root.tag.split("}")[-1]
    if root_tag != "svg":
        return False

    for elem in root.iter():
        tag = elem.tag.split("}")[-1]
        if tag not in ALLOWED_TAGS:
            return False

        for attr, val in elem.attrib.items():
            attr_l = attr.lower()
            val_l = str(val).lower()

            # no inline JS handlers
            if attr_l.startswith("on"):
                return False

            # no javascript URLs or external refs
            if "javascript:" in val_l:
                return False
            if "http://" in val_l or "https://" in val_l:
                return False

    return True

def sanitize_svg(svg_text: str) -> str:
    """
    Light post-processing.
    Keep it simple for now. If invalid, caller can fallback.
    """
    if not svg_text:
        return ""
    svg_text = svg_text.strip()
    return svg_text


In [11]:
def format_svg_sample(prompt: str, svg_code: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": svg_code},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


In [12]:
from datasets import Dataset

In [13]:
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Debug on a smaller subset first
#remember to uncomment/comment this for training set size
if CONFIG["debug_train_rows"] is not None:
    train_df = train_df.iloc[:CONFIG["debug_train_rows"]].copy()

eval_n = int(len(train_df) * CONFIG["eval_ratio"])
eval_df = train_df.iloc[:eval_n].copy()
train_df_small = train_df.iloc[eval_n:].copy()

print("Train rows:", len(train_df_small))
print("Eval rows:", len(eval_df))

# Build text column for SFT
train_df_small["text"] = train_df_small.apply(
    lambda row: format_svg_sample(row["prompt"], row["svg"]),
    axis=1,
)
eval_df["text"] = eval_df.apply(
    lambda row: format_svg_sample(row["prompt"], row["svg"]),
    axis=1,
)

train_ds = Dataset.from_pandas(train_df_small[["text"]], preserve_index=False)
eval_ds = Dataset.from_pandas(eval_df[["text"]], preserve_index=False)

print(train_ds[0]["text"][:500])

Train rows: 4750
Eval rows: 250
<|im_start|>system
You generate compact, valid SVG markup with Canvas size 256 × 256 pixels and at most 16,000 characters. Return only SVG code with a single root <svg> element. Max path count must be less than 256. Use only allowed SVG tags and keep the output concise. <|im_end|>
<|im_start|>user
A bright green circular icon featuring a golf hole with a flagpole and a frowning face, including a pair of eyes, a downturned mouth, and two cloud-like shapes above the head.<|im_end|>
<|im_start|>ass


In [14]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model for training...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading base model for training...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [16]:
from trl import SFTTrainer, SFTConfig

In [17]:
#SFTConfig instead of SFTTrainer

training_args = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    logging_steps=CONFIG["logging_steps"],
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    eval_strategy="steps",
    save_strategy="steps",
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    report_to="none",
    seed=SEED,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    
    dataset_text_field="text",
    max_length=CONFIG["max_seq_length"],   #  max_seq_length
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

train_result = trainer.train()
train_result

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/4750 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4750 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4750 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/250 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,0.416209,0.390016,0.404637,2479476.000000,0.868401
200,0.369178,0.362977,0.363603,4952073.000000,0.876588


TrainOutput(global_step=297, training_loss=0.41505787749884504, metrics={'train_runtime': 4914.4272, 'train_samples_per_second': 0.967, 'train_steps_per_second': 0.06, 'total_flos': 7.110643797570355e+16, 'train_loss': 0.41505787749884504, 'entropy': 0.36376725871253895, 'num_tokens': 7343859.0, 'mean_token_accuracy': 0.8798105906557154, 'epoch': 1.0})

In [19]:
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])

print(f"Saved adapter + tokenizer to: {CONFIG['output_dir']}")

Saved adapter + tokenizer to: ./svg_lora_adapter


In [20]:
# Option (a): Load adapter
print("Loading base model for inference...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
    trust_remote_code=True,
)

print("Loading LoRA adapter for inference...")
model = PeftModel.from_pretrained(base_model, CONFIG["output_dir"])

#reset just for safty
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

model.eval()
print("Model loaded for inference.")

Loading base model for inference...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading LoRA adapter for inference...
Model loaded for inference.


In [26]:
# =========================================================
def build_messages(user_prompt: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

def generate_svg(prompt: str, max_new_tokens: int = 400) -> str:
    messages = build_messages(prompt)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,   # deterministic first
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    # Only decode newly generated tokens
    new_token_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(new_token_ids, skip_special_tokens=True)

    svg = extract_svg(response)
    svg = sanitize_svg(svg)

    if not is_valid_svg(svg):
        svg = fallback_svg()

    return svg

#Disclose AI tooling, I used ChatGPT to learn how to see progress bar
def generate_svg_batch(prompts, batch_size=4, max_new_tokens=400):
    outputs = []

    with tqdm(total=len(prompts), desc="Generating SVG rows") as pbar:
        for i in range(0, len(prompts), batch_size):
            batch_prompts = prompts[i:i + batch_size]

            batch_texts = []
            for p in batch_prompts:
                messages = build_messages(p)
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
                batch_texts.append(text)

            inputs = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(model.device)

            with torch.no_grad():
                generated = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True,
                )

            for j in range(len(batch_prompts)):
                input_len = int(inputs["attention_mask"][j].sum().item())
                new_token_ids = generated[j][input_len:]
                response = tokenizer.decode(new_token_ids, skip_special_tokens=True)

                svg = extract_svg(response)
                svg = sanitize_svg(svg)

                if not is_valid_svg(svg):
                    svg = fallback_svg()

                outputs.append(svg)

            pbar.update(len(batch_prompts))

    return outputs

In [22]:
test_prompt = "a simple blue bird icon"
pred_svg = generate_svg(test_prompt)
print(pred_svg[:500])
print("Valid SVG:", is_valid_svg(pred_svg))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"><rect width="256" height="256" fill="white"/><circle cx="128" cy="128" r="48" fill="black"/></svg>
Valid SVG: True


In [25]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

In [28]:
#Testing on a small set

# small_prompts = test_df["prompt"].tolist()[:20]
# t0 = time.time()
# _ = generate_svg_batch(small_prompts, batch_size=4, max_new_tokens=400)
# print("Minutes for 20 rows:", (time.time() - t0) / 60)

Generating SVG rows:   0%|          | 0/20 [00:00<?, ?it/s]

Minutes for 20 rows: 1.9559730609258017


In [29]:
# Submission generation scaffold: expects Kaggle prompt file with columns `id,prompt`.
#TEST_PROMPTS_PATH = "/kaggle/input/svg-test-public-prompts/test_prompts.csv"
SUBMISSION_PATH = CONFIG["submission_path"]

rows = []
invalid_count = 0
t0 = time.time()

prompts = test_df["prompt"].tolist()
ids = test_df["id"].tolist()

pred_svgs = generate_svg_batch(prompts, batch_size=4, max_new_tokens=400)

for id_, svg in zip(ids, pred_svgs):
    if not is_valid_svg(svg):
        invalid_count += 1
        svg = fallback_svg()
    rows.append({"id": id_, "svg": svg})



Generating SVG rows:   0%|          | 0/1000 [00:00<?, ?it/s]

In [30]:
sub_df = pd.DataFrame(rows)
sub_df.to_csv(SUBMISSION_PATH, index=False)

elapsed_min = (time.time() - t0) / 60
print(f"Saved: {SUBMISSION_PATH}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid/fallback count: {invalid_count}")
print(f"Runtime (minutes): {elapsed_min:.2f}")
print(sub_df.head())

Saved: ./submission.csv
Rows: 1000
Invalid/fallback count: 0
Runtime (minutes): 100.68
                                     id  \
0  fa1d8fa7-080f-4269-a9cf-a17562c9a0ca   
1      6eede943219547c22ac56085027d33cc   
2      ea045c7a247166f061ce504d9b7ccaab   
3      8fe82f3af89e487b31236ca829c3f071   
4      600464e4d92c75338462271a09b3f176   

                                                 svg  
0  <svg xmlns="http://www.w3.org/2000/svg" width=...  
1  <svg xmlns="http://www.w3.org/2000/svg" width=...  
2  <svg xmlns="http://www.w3.org/2000/svg" width=...  
3  <svg xmlns="http://www.w3.org/2000/svg" width=...  
4  <svg xmlns="http://www.w3.org/2000/svg" width=...  


In [31]:
import shutil

zip_base = "./svg_lora_adapter"
if os.path.exists(CONFIG["output_dir"]):
    shutil.make_archive(zip_base, "zip", CONFIG["output_dir"])
    print("Zipped adapter folder:", zip_base + ".zip")

Zipped adapter folder: ./svg_lora_adapter.zip


In [36]:
import shutil

dst = os.path.expanduser("~/Downloads/submission.csv")
shutil.copy("submission.csv", dst)

print("Saved to:", dst)

Saved to: C:\Users\keyud/Downloads/submission.csv


## Notes

- Keep a fixed seed, runtime logs, and invalid-generation counts (required by `contest_docs/05`).
- If you use Kaggle-packaged datasets (`svg-train-public`, `svg-test-public-prompts`, `svg-utils`), swap paths into the loading cells.
- For stricter alignment with Unsloth templates, copy the latest prompt-formatting snippets from the official Qwen3.5-2B notebook linked above.